# Lesson 1a: Markov Decision Processes - Theory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powell-clark/reinforcement-learning/blob/main/notebooks/1a_mdp_theory.ipynb)

**Course**: Reinforcement Learning from First Principles  
**Lesson**: 1a - Markov Decision Processes (Theory)  
**Prerequisites**: Lesson 0a, 0b  
**Updated**: 2025

---

## Learning Objectives

By the end of this notebook, you will master:
1. **Formal MDP definition** - The mathematical framework for RL
2. **Markov Property** - Why memoryless states matter
3. **Bellman Equations** - The fundamental recursive structure
4. **Optimal Policies** - What does "optimal" mean mathematically?
5. **Value Functions** - V(s) and Q(s,a) from first principles
6. **Policy Evaluation** - Computing value functions analytically

## Table of Contents

1. [Introduction: Why MDPs?](#1-introduction)
2. [The Markov Property](#2-markov-property)
3. [Formal MDP Definition](#3-mdp-definition)
4. [Policies and Returns](#4-policies)
5. [Value Functions](#5-value-functions)
6. [Bellman Equations](#6-bellman-equations)
7. [Optimal Policies](#7-optimal-policies)
8. [Simple MDP Example](#8-example)
9. [Summary](#9-summary)

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Tuple, List
import itertools

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

SEED = 42
np.random.seed(SEED)

print("✓ Imports complete")

## 1. Introduction: Why MDPs?

Markov Decision Processes provide the **mathematical foundation** for reinforcement learning. Every RL problem can be formulated as an MDP (or approximated as one).

### Historical Context

- **1950s**: Richard Bellman develops Dynamic Programming
- **1960s**: Howard introduces policy iteration
- **1980s**: MDPs become foundation for modern RL
- **Today**: All RL algorithms trace back to MDP theory

### Why This Matters

Understanding MDPs gives you:
- **Theoretical foundation** for all RL algorithms
- **Proof techniques** for convergence guarantees
- **Design principles** for new algorithms
- **Debug intuition** when algorithms fail

### Real-World MDP Applications

- **Games**: Chess positions = states, moves = actions
- **Robotics**: Joint angles = states, motor commands = actions
- **Finance**: Portfolio = state, trades = actions
- **LLMs**: Token sequence = state, next token = action

## 2. The Markov Property

The **Markov Property** is the foundation of MDPs. It states:

> The future is independent of the past given the present.

### Mathematical Definition

A state $S_t$ is **Markov** if and only if:

$$P(S_{t+1} | S_t, S_{t-1}, ..., S_0) = P(S_{t+1} | S_t)$$

In words: The probability of the next state depends only on the current state, not on the history of how we got here.

### Intuition

The current state contains **all relevant information** for predicting the future. The history is irrelevant if we know the current state.

### Examples

**Markov ✓**:
- Chess board position (contains all information)
- Robot joint positions + velocities (full state)
- Stock price with market indicators (if comprehensive)

**Not Markov ✗**:
- Stock price alone (missing momentum, trends)
- Single frame of Pong (missing ball velocity)
- Robot position without velocity (missing dynamics)

### Making Non-Markov Markov

**Solution**: Augment state with history!
- Single frame → Stack last 4 frames
- Position → (Position, Velocity)
- Price → (Price, Moving Average, Volume)

This creates a **Markov state** from non-Markov observations.

In [ ]:
# Demonstration: Markov vs Non-Markov
print("=" * 70)
print("DEMONSTRATION: Markov Property")
print("=" * 70 + "\n")

# Markov chain: Each state's next state depends only on current state
# States: 0, 1, 2 (e.g., weather: Sunny, Cloudy, Rainy)
transition_matrix = np.array([
    [0.7, 0.2, 0.1],  # From Sunny: 70% stay sunny, 20% cloudy, 10% rainy
    [0.3, 0.4, 0.3],  # From Cloudy
    [0.2, 0.3, 0.5],  # From Rainy
])

states = ['Sunny', 'Cloudy', 'Rainy']

# Simulate Markov chain
current_state = 0  # Start sunny
history = [current_state]

for t in range(10):
    # Next state depends ONLY on current state (Markov property)
    probs = transition_matrix[current_state]
    current_state = np.random.choice([0, 1, 2], p=probs)
    history.append(current_state)

print("Markov Chain Simulation (Weather):")
print("State sequence:", ' → '.join([states[s] for s in history]))

# Verify Markov property
print("\nVerifying Markov Property:")
print(f"Given current state is {states[history[-1]]}")
print(f"P(next=Sunny | current={states[history[-1]]}) = {transition_matrix[history[-1]][0]:.1f}")
print(f"This probability doesn't depend on history before current state!")

print("\n✓ Markov property: Future depends only on present, not past")

## 3. Formal MDP Definition

A **Markov Decision Process** is a tuple $(S, A, P, R, \gamma)$ where:

### Components

1. **$S$**: State space (finite or infinite set of states)
   - Example: Chess positions, robot configurations

2. **$A$**: Action space (finite or infinite set of actions)
   - Example: Chess moves, motor torques

3. **$P$**: Transition probability function
   $$P(s' | s, a) = P(S_{t+1} = s' | S_t = s, A_t = a)$$
   - Probability of reaching state $s'$ from state $s$ via action $a$

4. **$R$**: Reward function
   $$R(s, a, s') = E[R_{t+1} | S_t = s, A_t = a, S_{t+1} = s']$$
   - Expected immediate reward for transition
   - Often simplified: $R(s,a)$ or $R(s)$

5. **$\gamma$**: Discount factor, $\gamma \in [0,1]$
   - Controls importance of future rewards
   - $\gamma=0$: Only immediate reward
   - $\gamma=1$: All future rewards equally important

### Dynamics

An MDP evolves as:

$$S_0 \xrightarrow{A_0} R_1, S_1 \xrightarrow{A_1} R_2, S_2 \xrightarrow{A_2} R_3, S_3 \xrightarrow{A_3} ...$$

At each time $t$:
1. Agent observes state $S_t$
2. Agent selects action $A_t$
3. Environment returns reward $R_{t+1}$
4. Environment transitions to state $S_{t+1}$ according to $P(\cdot|S_t, A_t)$

### Example: GridWorld MDP

- **States**: Grid cells $(i,j)$
- **Actions**: {Up, Down, Left, Right}
- **Transitions**: Deterministic movement (or stochastic with slip)
- **Rewards**: -1 per step, +10 at goal, -5 in holes
- **Discount**: $\gamma = 0.99$

In [ ]:
# Simple MDP Implementation
class SimpleMDP:
    """
    Simple MDP for demonstration.
    3-state chain: s0 → s1 → s2 (terminal)
    Actions: 0=stay, 1=advance
    """
    
    def __init__(self):
        self.n_states = 3
        self.n_actions = 2
        self.gamma = 0.9
        
        # Define transition probabilities P[s][a][s']
        self.P = np.zeros((self.n_states, self.n_actions, self.n_states))
        
        # State 0
        self.P[0][0][0] = 1.0  # Stay → stay at 0
        self.P[0][1][1] = 1.0  # Advance → go to 1
        
        # State 1
        self.P[1][0][1] = 1.0  # Stay → stay at 1
        self.P[1][1][2] = 1.0  # Advance → go to 2 (terminal)
        
        # State 2 (terminal)
        self.P[2][0][2] = 1.0  # Absorbing state
        self.P[2][1][2] = 1.0
        
        # Define rewards R[s][a]
        self.R = np.array([
            [-1, -1],  # State 0: -1 for any action
            [-1, 5],   # State 1: -1 for stay, +5 for advancing to goal
            [0, 0],    # State 2: terminal, no more rewards
        ])
    
    def get_transition_prob(self, s, a, s_next):
        """Get P(s'|s,a)"""
        return self.P[s][a][s_next]
    
    def get_reward(self, s, a):
        """Get R(s,a)"""
        return self.R[s][a]
    
    def is_terminal(self, s):
        """Check if state is terminal"""
        return s == 2

# Create and examine MDP
mdp = SimpleMDP()

print("Simple MDP Structure:")
print(f"States: {mdp.n_states} (s0, s1, s2)")
print(f"Actions: {mdp.n_actions} (0=stay, 1=advance)")
print(f"Discount factor: γ = {mdp.gamma}")

print("\nTransition Probabilities P(s'|s,a):")
for s in range(mdp.n_states):
    for a in range(mdp.n_actions):
        for s_next in range(mdp.n_states):
            prob = mdp.P[s][a][s_next]
            if prob > 0:
                action_name = 'stay' if a == 0 else 'advance'
                print(f"  P(s{s_next} | s{s}, {action_name}) = {prob}")

print("\nReward Function R(s,a):")
for s in range(mdp.n_states):
    for a in range(mdp.n_actions):
        action_name = 'stay' if a == 0 else 'advance'
        print(f"  R(s{s}, {action_name}) = {mdp.R[s][a]}")

print("\n✓ MDP formally defined!")

## 4. Policies and Returns

### Policy Definition

A **policy** $\pi$ specifies what action to take in each state.

**Deterministic Policy**:
$$\pi: S \rightarrow A$$
$$a = \pi(s)$$

**Stochastic Policy**:
$$\pi: S \times A \rightarrow [0,1]$$
$$\pi(a|s) = P(A_t = a | S_t = s)$$

### Return (Cumulative Reward)

The **return** $G_t$ is the total discounted reward from time $t$:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

### Why Discounting?

1. **Mathematical convenience**: Ensures finite sums
2. **Uncertainty**: Future is less predictable
3. **Preference**: Immediate rewards preferred
4. **Cognitive modeling**: Matches animal/human behavior

### Recursive Property

Key insight: Returns have recursive structure!

$$G_t = R_{t+1} + \gamma G_{t+1}$$

This recursive property is the foundation of **Bellman equations**.

In [ ]:
# Computing returns
def compute_return(rewards, gamma=0.9):
    """
    Compute discounted return from reward sequence.
    
    G = r1 + γ*r2 + γ²*r3 + ...
    """
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
    return G

# Example reward sequences
print("Computing Returns with Different Discount Factors:\n")

rewards = [1, 1, 1, 1, 1]  # Constant rewards

for gamma in [0.0, 0.5, 0.9, 0.99, 1.0]:
    G = compute_return(rewards, gamma)
    print(f"γ = {gamma:.2f}: G = {G:.3f}")

print("\nInterpretation:")
print("- γ=0.0: Only immediate reward matters (G=1)")
print("- γ=1.0: All future rewards equal (G=5)")
print("- γ=0.9: Balanced discount (G≈4.095)")

# Verify recursive property
print("\n" + "="*70)
print("Verifying Recursive Property: G_t = R_{t+1} + γ*G_{t+1}")
print("="*70)

rewards = [1, 2, 3, 4, 5]
gamma = 0.9

# Method 1: Direct computation
G_direct = compute_return(rewards, gamma)

# Method 2: Recursive
G_future = compute_return(rewards[1:], gamma)  # G_{t+1}
G_recursive = rewards[0] + gamma * G_future    # R_{t+1} + γ*G_{t+1}

print(f"Direct computation: G = {G_direct:.6f}")
print(f"Recursive: R + γ*G_next = {G_recursive:.6f}")
print(f"Match: {np.isclose(G_direct, G_recursive)}")

print("\n✓ Recursive property verified!")

## 5. Value Functions

Value functions are the **core** of RL. They predict long-term reward.

### State-Value Function

The **state-value function** $V^\pi(s)$ is the expected return starting from state $s$ and following policy $\pi$:

$$V^\pi(s) = E_\pi[G_t | S_t = s]$$
$$= E_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t = s\right]$$

In words: "How good is it to be in state $s$ under policy $\pi$?"

### Action-Value Function

The **action-value function** $Q^\pi(s,a)$ is the expected return starting from state $s$, taking action $a$, then following policy $\pi$:

$$Q^\pi(s,a) = E_\pi[G_t | S_t = s, A_t = a]$$
$$= E_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t = s, A_t = a\right]$$

In words: "How good is it to take action $a$ in state $s$, then follow policy $\pi$?"

### Relationship

$$V^\pi(s) = \sum_a \pi(a|s) Q^\pi(s,a)$$

The value of a state is the weighted average of Q-values, weighted by the policy.

$$Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s')$$

The Q-value is immediate reward plus discounted value of next state.

## 6. Bellman Equations

The **Bellman equations** are the fundamental recursive relationships in RL.

### Bellman Expectation Equation for $V^\pi$

**Derivation**:
$$V^\pi(s) = E_\pi[G_t | S_t = s]$$
$$= E_\pi[R_{t+1} + \gamma G_{t+1} | S_t = s]$$
$$= \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[R(s,a,s') + \gamma E_\pi[G_{t+1} | S_{t+1} = s']\right]$$
$$= \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[R(s,a,s') + \gamma V^\pi(s')\right]$$

**Simplified** (when $R$ doesn't depend on $s'$):
$$\boxed{V^\pi(s) = \sum_a \pi(a|s) \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s')\right]}$$

### Bellman Expectation Equation for $Q^\pi$

$$\boxed{Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \sum_{a'} \pi(a'|s') Q^\pi(s',a')}$$

### Intuition

Bellman equations say:
> The value of a state equals the immediate reward plus the discounted value of the next state.

This is a **system of linear equations** that we can solve!

### Matrix Form

For finite MDPs with deterministic policy:
$$V^\pi = R^\pi + \gamma P^\pi V^\pi$$
$$V^\pi = (I - \gamma P^\pi)^{-1} R^\pi$$

This gives us an **exact solution** for small MDPs!

In [ ]:
# Solving Bellman equation analytically
def solve_bellman_analytical(mdp, policy):
    """
    Solve Bellman equation: V = R + γPV
    Solution: V = (I - γP)^{-1} R
    
    Args:
        mdp: MDP object
        policy: Deterministic policy (array of actions for each state)
    
    Returns:
        V: State values
    """
    n = mdp.n_states
    
    # Build P^π and R^π for the policy
    P_pi = np.zeros((n, n))
    R_pi = np.zeros(n)
    
    for s in range(n):
        a = policy[s]
        R_pi[s] = mdp.get_reward(s, a)
        for s_next in range(n):
            P_pi[s][s_next] = mdp.get_transition_prob(s, a, s_next)
    
    # Solve: V = (I - γP)^{-1} R
    I = np.eye(n)
    V = np.linalg.solve(I - mdp.gamma * P_pi, R_pi)
    
    return V

# Test on simple MDP
print("="*70)
print("SOLVING BELLMAN EQUATION ANALYTICALLY")
print("="*70 + "\n")

mdp = SimpleMDP()

# Policy 1: Always stay
policy_stay = np.array([0, 0, 0])  # Always action 0 (stay)
V_stay = solve_bellman_analytical(mdp, policy_stay)

print("Policy 1: Always STAY")
for s in range(mdp.n_states):
    print(f"  V(s{s}) = {V_stay[s]:.3f}")

# Policy 2: Always advance
policy_advance = np.array([1, 1, 1])  # Always action 1 (advance)
V_advance = solve_bellman_analytical(mdp, policy_advance)

print("\nPolicy 2: Always ADVANCE")
for s in range(mdp.n_states):
    print(f"  V(s{s}) = {V_advance[s]:.3f}")

print("\nAnalysis:")
print(f"  'Advance' policy is better: V(s0) = {V_advance[0]:.3f} > {V_stay[0]:.3f}")
print("  This makes sense: advancing gets reward +5 at s1→s2")

print("\n✓ Bellman equation solved analytically!")

## 7. Optimal Policies and Bellman Optimality

### Optimal Policy

A policy $\pi^*$ is **optimal** if:
$$V^{\pi^*}(s) \geq V^\pi(s) \quad \forall s \in S, \forall \pi$$

**Theorem**: For any MDP, there exists at least one optimal policy.

### Optimal Value Functions

$$V^*(s) = \max_\pi V^\pi(s)$$
$$Q^*(s,a) = \max_\pi Q^\pi(s,a)$$

### Bellman Optimality Equations

**For $V^*$**:
$$\boxed{V^*(s) = \max_a \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s')\right]}$$

**For $Q^*$**:
$$\boxed{Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \max_{a'} Q^*(s',a')}$$

### Relationship:
$$V^*(s) = \max_a Q^*(s,a)$$
$$Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s')$$

### Optimal Policy from $Q^*$:
$$\pi^*(s) = \arg\max_a Q^*(s,a)$$

### Key Difference

- **Expectation equations** (Bellman): Linear system, can solve directly
- **Optimality equations**: Nonlinear due to $\max$, need iterative methods

In [ ]:
# Value Iteration: Solve Bellman Optimality Equation
def value_iteration(mdp, theta=1e-6, max_iterations=1000):
    """
    Solve Bellman optimality equation using value iteration.
    
    Iterates: V(s) ← max_a [R(s,a) + γ Σ P(s'|s,a) V(s')]
    
    Args:
        mdp: MDP object
        theta: Convergence threshold
        max_iterations: Maximum iterations
    
    Returns:
        V: Optimal state values
        policy: Optimal policy
    """
    V = np.zeros(mdp.n_states)
    
    for iteration in range(max_iterations):
        delta = 0
        
        for s in range(mdp.n_states):
            if mdp.is_terminal(s):
                continue
            
            v = V[s]
            
            # Compute max over actions
            action_values = []
            for a in range(mdp.n_actions):
                # Q(s,a) = R(s,a) + γ Σ P(s'|s,a) V(s')
                q_value = mdp.get_reward(s, a)
                for s_next in range(mdp.n_states):
                    prob = mdp.get_transition_prob(s, a, s_next)
                    q_value += mdp.gamma * prob * V[s_next]
                action_values.append(q_value)
            
            V[s] = max(action_values)
            delta = max(delta, abs(v - V[s]))
        
        if delta < theta:
            print(f"Converged in {iteration + 1} iterations")
            break
    
    # Extract optimal policy
    policy = np.zeros(mdp.n_states, dtype=int)
    for s in range(mdp.n_states):
        action_values = []
        for a in range(mdp.n_actions):
            q_value = mdp.get_reward(s, a)
            for s_next in range(mdp.n_states):
                prob = mdp.get_transition_prob(s, a, s_next)
                q_value += mdp.gamma * prob * V[s_next]
            action_values.append(q_value)
        policy[s] = np.argmax(action_values)
    
    return V, policy

# Solve for optimal policy
print("="*70)
print("FINDING OPTIMAL POLICY VIA VALUE ITERATION")
print("="*70 + "\n")

V_optimal, policy_optimal = value_iteration(mdp)

print("\nOptimal State Values V*(s):")
for s in range(mdp.n_states):
    print(f"  V*(s{s}) = {V_optimal[s]:.3f}")

print("\nOptimal Policy π*(s):")
actions = ['stay', 'advance']
for s in range(mdp.n_states):
    print(f"  π*(s{s}) = {actions[policy_optimal[s]]}")

print("\nVerification:")
print("  Optimal policy says: always ADVANCE")
print("  This maximizes long-term reward!")

print("\n✓ Optimal policy found!")

## 8. Complete Example: Student MDP

Let's solve a complete MDP from scratch.

### Problem: Student Study Decision

**States**:
- 0: Tired
- 1: Rested
- 2: Exam (terminal)

**Actions**:
- 0: Sleep
- 1: Study

**Transitions**:
- Tired + Sleep → Rested (certain)
- Tired + Study → Tired (stay tired)
- Rested + Sleep → Rested (stay rested)
- Rested + Study → 70% Exam, 30% Tired

**Rewards**:
- Sleep: 0
- Study when Tired: -2 (ineffective)
- Study when Rested: +10 if leads to exam, 0 otherwise

**Goal**: Find optimal policy!

In [ ]:
# Student MDP
class StudentMDP:
    def __init__(self):
        self.n_states = 3  # Tired, Rested, Exam
        self.n_actions = 2  # Sleep, Study
        self.gamma = 0.9
        
        # Transitions P[s][a][s']
        self.P = np.zeros((3, 2, 3))
        self.P[0][0][1] = 1.0  # Tired + Sleep → Rested
        self.P[0][1][0] = 1.0  # Tired + Study → Tired
        self.P[1][0][1] = 1.0  # Rested + Sleep → Rested
        self.P[1][1][2] = 0.7  # Rested + Study → Exam (70%)
        self.P[1][1][0] = 0.3  # Rested + Study → Tired (30%)
        self.P[2][0][2] = 1.0  # Exam is absorbing
        self.P[2][1][2] = 1.0
        
        # Rewards R[s][a]
        self.R = np.array([
            [0, -2],   # Tired: sleep=0, study=-2 (inefficient)
            [0, 7],    # Rested: sleep=0, study=7 (0.7*10)
            [0, 0],    # Exam: terminal
        ])
    
    def get_transition_prob(self, s, a, s_next):
        return self.P[s][a][s_next]
    
    def get_reward(self, s, a):
        return self.R[s][a]
    
    def is_terminal(self, s):
        return s == 2

# Solve Student MDP
print("="*70)
print("STUDENT MDP: Finding Optimal Study Strategy")
print("="*70 + "\n")

student_mdp = StudentMDP()
V_opt, pi_opt = value_iteration(student_mdp)

states = ['Tired', 'Rested', 'Exam']
actions = ['Sleep', 'Study']

print("\nOptimal Values:")
for s in range(3):
    print(f"  V*({states[s]}) = {V_opt[s]:.3f}")

print("\nOptimal Policy:")
for s in range(3):
    print(f"  When {states[s]}: {actions[pi_opt[s]]}")

print("\nInterpretation:")
print("  When Tired: Sleep (to recover)")
print("  When Rested: Study (effective study)")
print("  This maximizes probability of reaching exam successfully!")

print("\n✓ Optimal study strategy found!")

## 9. Summary

### What We Learned

1. **Markov Property**: Future independent of past given present
2. **MDP Definition**: $(S, A, P, R, \gamma)$
3. **Policies**: Deterministic $\pi(s)$ and stochastic $\pi(a|s)$
4. **Returns**: $G_t = \sum_{k=0}^\infty \gamma^k R_{t+k+1}$
5. **Value Functions**: $V^\pi(s)$ and $Q^\pi(s,a)$
6. **Bellman Equations**: Recursive structure of values
7. **Optimality**: $V^*(s) = \max_a Q^*(s,a)$
8. **Solution Methods**: Analytical and iterative

### Key Equations

**Bellman Expectation**:
$$V^\pi(s) = \sum_a \pi(a|s) \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s')\right]$$

**Bellman Optimality**:
$$V^*(s) = \max_a \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s')\right]$$

### Why This Matters

- **Every RL algorithm** is solving these equations (approximately)
- **Q-learning, SARSA, DQN, PPO** all trace back to Bellman equations
- Understanding MDPs gives **theoretical foundation** for all RL

### Next Steps

**Next**: [1b: MDP Practical](./1b_mdp_practical.ipynb)
- Policy evaluation implementation
- Policy iteration algorithm
- Value iteration on FrozenLake

Then: [2a: Dynamic Programming](./2a_dynamic_programming_theory.ipynb)
- DP methods for solving MDPs
- Asynchronous DP
- Generalized policy iteration

### Exercises

1. Prove that optimal policy exists for finite MDPs
2. Derive Bellman equation for $Q^\pi(s,a)$ from scratch
3. Implement policy evaluation using Bellman equation
4. Create your own MDP and solve it
5. Show that $\gamma=1$ can lead to infinite values

---

**Congratulations!** You now understand the mathematical foundation of reinforcement learning.

**Next**: [Lesson 1b: MDP Practical →](./1b_mdp_practical.ipynb)